In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 트랜스파일에 자주 사용되는 파라미터

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
아래 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
이 페이지에서는 로컬 트랜스파일에 자주 사용되는 파라미터 중 일부를 설명합니다. 이 파라미터들은 [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) 또는 [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile)에 인수를 전달하여 설정합니다.

<span id="approx-degree"></span>
## 근사 정도
근사 정도(approximation degree)를 사용하여 결과 Circuit이 원하는(입력) Circuit과 얼마나 가깝게 일치해야 하는지를 지정할 수 있습니다. 이 값은 (0.0 - 1.0) 범위의 부동소수점으로, 0.0은 최대 근사이고 1.0(기본값)은 근사 없음을 의미합니다. 값이 작을수록 출력 정확도를 희생하는 대신 실행이 쉬워집니다(즉, 게이트 수가 줄어듭니다). 기본값은 1.0입니다.

2-Qubit 유니터리 합성(모든 레벨의 초기 단계 및 최적화 레벨 3의 최적화 단계에서 사용됨)에서 이 값은 출력 분해의 목표 충실도를 지정합니다. 즉, Circuit의 행렬 표현이 이산 Gate로 변환될 때 도입되는 오류의 양입니다. 근사 정도가 낮을수록(더 많은 근사) 합성으로 생성된 출력 Circuit은 입력 행렬과 더 많이 달라지지만, 게이트 수는 더 적어질 가능성이 높습니다(임의의 2-Qubit 연산은 최대 세 개의 CX Gate로 완벽하게 분해될 수 있기 때문입니다). 이는 실행이 더 쉬워집니다.

근사 정도가 1.0 미만인 경우, 하나 또는 두 개의 CX Gate를 가진 Circuit이 합성될 수 있습니다. 이로 인해 하드웨어에서 발생하는 오류는 줄어들지만 근사에서 발생하는 오류는 늘어납니다. CX는 오류 측면에서 가장 비용이 높은 Gate이기 때문에, 합성 충실도를 희생하더라도 그 수를 줄이는 것이 유리할 수 있습니다(이 기법은 IBM&reg; 장치에서 양자 볼륨을 높이는 데 사용되었습니다: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)).

예시로, 초기 단계에서 합성될 임의의 2-Qubit `UnitaryGate`를 생성합니다. `approximation_degree`를 1.0 미만으로 설정하면 근사 Circuit이 생성될 수 있습니다. 합성 방법이 어떤 Gate를 사용할 수 있는지 알 수 있도록 `basis_gates`도 함께 지정해야 합니다.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

근사로 인해 더 적은 CX Gate가 필요하므로 출력은 `2`가 됩니다.

<span id="seed"></span>
## 난수 생성기 시드
Transpiler의 일부는 확률적으로 동작하므로, 반복 실행 시 다른 결과가 반환될 수 있습니다. 재현 가능한 결과를 얻으려면 `seed_transpiler` 인수를 사용하여 의사 난수 생성기의 시드를 설정할 수 있습니다. 동일한 시드를 사용하여 반복 실행하면 동일한 결과가 반환됩니다.

예시:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## 초기 레이아웃
트랜스파일 전에 Circuit에 포함된 Qubit는 대상 Backend의 물리적 Qubit에 반드시 대응되지 않는 가상 Qubit입니다. `initial_layout` 인수를 사용하여 가상 Qubit을 물리적 Qubit에 초기 매핑하는 방법을 지정할 수 있습니다. 최종 Qubit 레이아웃은 Transpiler가 스왑 Gate나 다른 방법으로 Qubit를 재배치할 수 있기 때문에 초기 레이아웃과 다를 수 있습니다.

아래 예시에서는 [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout) 객체를 생성하여 [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) 모의 Backend에 대한 초기 레이아웃을 구성합니다. 이 레이아웃은 Circuit의 첫 번째 Qubit을 Sherbrooke의 Qubit 5에, 두 번째 Qubit을 Sherbrooke의 Qubit 6에 매핑합니다. 물리적 Qubit은 항상 정수로 표현됩니다.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

Layout 객체를 지정하는 것 외에, 정수 리스트를 전달할 수도 있습니다. 이 경우 리스트의 $i$번째 요소에는 $i$번째 Qubit이 매핑될 물리적 Qubit이 포함됩니다. 예시:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

[`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) 함수를 사용하면 오류 정보와 물리적 Qubit 레이블이 표시된 장치 그래프 다이어그램을 생성할 수 있습니다. [Compute resources](https://quantum.cloud.ibm.com/computers) 페이지에서도 유사한 다이어그램을 확인할 수 있습니다.